In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os

sys.path.append(os.path.abspath(".."))
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scripts.TPS import ThinPlateSpline
from scripts.plotting import *

# ---------- Load vector field ----------
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time

# ---------- File organization ----------
path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv"
}

# Custom layout
row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

# ---------- 2×4 grid plot ----------
fig, axs = plt.subplots(2, 4, figsize=(20, 10))

for i, (ax, name) in enumerate(zip(axs.ravel(), plot_order)):
    X, V, time = load_vector_field(path_map[name])
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize time globally

    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    if i == 0:
        stream_density = 0.4
        aspect = 2.5
    elif i == 1:
        stream_density = 0.8
        aspect = 2.0
    elif i < 4:
        stream_density = 0.8
        aspect = 1.5
    else:
        stream_density = 1
        aspect = "equal"

    plot_velocity_streamplot(
        X_2d=X,
        tps_vf=tps_vf,
        grid_density=1.0, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=80,
        scatter_alpha=0.5,
        ax=ax,
        figsize=(5, 4),
        aspect=aspect,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        grid_size=50
    )
    ax.set_title(
        name.replace("_", " "),
        fontsize=36,
        pad=18)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Remove grid (in case streamplot turns it on)
    ax.grid(False)
    
    # Remove box / spines
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.manifold import MDS
from scipy.spatial.distance import squareform
from scripts.phase_distance_full_matrix_solver import *
from scipy.sparse.csgraph import shortest_path
import umap


np.random.seed(42)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

simulation_results = {}
noise = 0.001
extra_dim = 0

for i, (name, path) in enumerate(path_map.items()):
    # Load clean 2D data
    X_gt, V_gt, time = load_vector_field(path_map[name])

    # Add noise
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)

    # Add dummy dimensions
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))

    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    
    # X = (X - X.mean(axis=0)) / X.std(axis=0)
    # V = (V - V.mean(axis=0)) / V.std(axis=0)
    
    # --- compute full phase distance matrix ---
    solver = PhaseDistanceSolver(X, V)
    
    # phase_dist = solver.phase_ridge_distance(reg=0.5)
    phase_dist = solver.phase_constrained_L2_distance()
    
    # optional: clip extreme values for stability
    upper = phase_dist[np.triu_indices_from(phase_dist, k=1)]
    clip_val = np.quantile(upper, 0.99)
    phase_dist = np.minimum(phase_dist, clip_val)
    
    # --- MDS embedding ---
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        random_state=42,
        normalized_stress="auto"
    )
    
    embedding = mds.fit_transform(phase_dist)

    # Save all results
    simulation_results[name] = {
        "X": X,
        "V": V,
        "embedding": embedding,
        "true_time": time,
    }

    # Plot the embedding colored by true time
    ax = axes[i]
    sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=time, cmap='viridis', s=20)
    ax.set_title(
        name.replace("_", " "),
        fontsize=36,
        pad=18)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Remove grid (in case streamplot turns it on)
    ax.grid(False)
    
    # Remove box / spines
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from scripts.phase_distance_solver import PhaseDistanceGraphSolver
from scripts.phase_distance_full_matrix_solver import PhaseDistanceSolver


np.random.seed(42)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

simulation_results = {}
noise = 0.001
extra_dim = 0

for i, (name, path) in enumerate(path_map.items()):
    # --- Load clean data ---
    X_gt, V_gt, time = load_vector_field(path_map[name])

    # --- Add small noise ---
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)

    # --- Optional dummy dimensions ---
    if extra_dim > 0:
        X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
        V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))
        X = np.hstack([X_noisy, X_dummy])
        V = np.hstack([V_noisy, V_dummy])
    else:
        X, V = X_noisy, V_noisy

    # --------------------------------------------------
    # Graph-based phase distance (Isomap-style)
    # --------------------------------------------------
    solver = PhaseDistanceGraphSolver(
        X, V,
        k=15,          # larger k for global connectivity
        alpha=1.0
    )

    # Build phase-only graph from local τ estimates
    i_idx = solver.i_idx
    j_idx = solver.j_idx
    t_ls  = solver.t_ls

    phase_edge_dist = np.abs(t_ls[:, 0] - t_ls[:, 1]) + 1e-8

    # Symmetric sparse graph
    from scipy.sparse import coo_matrix
    n = X.shape[0]
    phase_graph = coo_matrix(
        (
            np.concatenate([phase_edge_dist, phase_edge_dist]),
            (
                np.concatenate([i_idx, j_idx]),
                np.concatenate([j_idx, i_idx])
            )
        ),
        shape=(n, n)
    )

    # --- Geodesic distances (Isomap step) ---
    geo_phase_dist = shortest_path(
        csgraph=phase_graph,
        directed=False,
        method="D"
    )

    # Optional clipping for numerical stability
    upper = geo_phase_dist[np.triu_indices_from(geo_phase_dist, k=1)]
    clip_val = np.quantile(upper[np.isfinite(upper)], 0.99)
    geo_phase_dist = np.minimum(geo_phase_dist, clip_val)

    # --- MDS embedding ---
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        random_state=42,
        normalized_stress="auto"
    )

    embedding = mds.fit_transform(geo_phase_dist)

    # Save results
    simulation_results[name] = {
        "X": X,
        "V": V,
        "embedding": embedding,
        "true_time": time,
    }

    # --- Plot ---
    ax = axes[i]
    sc = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=time,
        cmap="viridis",
        s=20
    )

    ax.set_title(
        name.replace("_", " "),
        fontsize=36,
        pad=18
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
phase_par = {
    "straight_line": {"center":[0,0], "reference_vector":[1,0]},
    "sine_curve": {"center":[0,0], "reference_vector":[0,-1]},
    "branch_2": {"center":[0,0], "reference_vector":[-1,0]},
    "branch_4": {"center":[0,0], "reference_vector":[-1,0]},
    "rotation": {"center":[0,0], "reference_vector":[1,0]},
    "spiral": {"center":[0,0], "reference_vector":[1,0]},
    "saddle": {"center":[1,1], "reference_vector":[0.1,-1]},
    "quadratic_source_sink": {"center":[-0.5,-0.5], "reference_vector":[-1,0]}
}

def compute_phase_from_reference(embedding, center, reference_vector):
    rel = embedding - center
    ref_angle = np.arctan2(reference_vector[1], reference_vector[0])
    raw_angle = np.arctan2(rel[:, 1], rel[:, 0])
    phase_angle = (raw_angle - ref_angle + 2 * np.pi) % (2 * np.pi)
    return phase_angle

In [ ]:
# Plotting
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (name, result) in enumerate(simulation_results.items()):
    embedding = result["embedding"]

    center = phase_par[name]["center"]
    ref_vec = phase_par[name]["reference_vector"]
    phase_angle = compute_phase_from_reference(embedding, center, ref_vec)

    # Save phase angle back to results
    simulation_results[name]["phase_angle"] = phase_angle

    # Plot colored by phase angle
    ax = axes[i]
    sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=phase_angle, cmap='twilight', s=20)
    ax.set_title(name.replace('_', ' ').title())
    ax.set_xlabel("MDS 1")
    ax.set_ylabel("MDS 2")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (name, result) in enumerate(simulation_results.items()):
    true_time = result["true_time"]
    phase_angle = result["phase_angle"]

    ax = axes[i]
    ax.scatter(true_time, phase_angle, s=10, alpha=0.8)
    ax.set_title(name.replace('_', ' ').title())
    ax.set_xlabel("True Time")
    ax.set_ylabel("Phase Angle (radians)")

plt.tight_layout()
plt.show()